In [1]:
import json
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict

from src.minbpe import RegexTokenizer
from src.gpt import GPTLanguageModel

import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

class FineTuningDataset(Dataset):
    def __init__(self, data: torch.Tensor, device: torch.device, padding_token: int):
        self.data = data
        self.device = device
        self.padding_token = padding_token

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sample = self.data[index]
        x = sample.to(self.device)
        y = sample[1:].to(self.device)
        padding_tensor = torch.tensor([self.padding_token], device=self.device)
        y = torch.cat((y, padding_tensor))

        return x, y

@torch.no_grad()
def estimate_loss(
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
) -> Dict[str, float]:
    output = {}
    model.eval()

    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = []
        for x, y in loader:
            with torch.no_grad():
                _, loss = model(x, y)
            losses.append(loss.item())
        output[split] = sum(losses) / len(losses)

    model.train()
    return output

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_dir = Path("data") / "tokenizer"

checkpoint_dir = Path("data") / "ch05_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [4]:
tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_tensor = torch.load(tokenizer_dir /'ch05_train.ft.pt')
val_tensor = torch.load(tokenizer_dir /'ch05_validation.ft.pt')

In [5]:
#block_size = 256 # ch02: 256, ch03: 512
#n_embd = 512
#n_head = 8
#n_layer = 4
#dropout = 0.2
#vocab_size = len(tokenizer.vocab)

padding_token = -100
learning_rate = 1e-4

batch_size = 64
iter_start, max_iters = 1, 60
eval_interval = 20

ckpt_files = sorted(
    checkpoint_dir.glob("checkpoint_*.pt"),
    key=lambda x: x.stat().st_ctime,
    #key=lambda x: int(x.name.split("-")[1]),
    reverse=True,
)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    iter_start = int(checkpoint_path.name.replace("checkpoint_", "").replace(".pt", "")) + 1
else:
    checkpoint_path = Path("data") / "ch02_checkpoints" / "checkpoint_001-377870.pt"

print(f"load checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)
parameters = checkpoint['meta']['parameters']

load checkpoint: data/ch05_checkpoints/checkpoint_000060.pt


In [6]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    ignore_index=tokenizer.special_tokens["<|padding|>"],
    device=device,
)

model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.load_state_dict(checkpoint["model_state_dict"])

num_parameters = sum(p.numel() for p in model.parameters()) / 1e6
print(f'--> {num_parameters:_.3}M parameters')
# print_model_structure(model)

--> 13.8M parameters


In [7]:
input_tokens = tokenizer.encode("hello, world", allowed_special="all")
input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)
    print(tokenizer.decode(output[0].tolist()))

_ = model.train()

hello, world!"

ChatGPT^n = more powerfully something - "what is your name?"

Please be more specific: Are there any Python libraries that could help me with this?<|endoftext|><|startoftext|>assistant<|separator|>__Specy____^_/ I want you to create an alternative python function that computes the OpenAI API key into


In [8]:
train_dataset = FineTuningDataset(data=train_tensor, device=device, padding_token=padding_token)
val_dataset = FineTuningDataset(data=val_tensor, device=device, padding_token=padding_token)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size)

train_losses, val_losses = [], []

In [9]:
total_steps = len(train_loader)
print(f"--> Start training: iter_start={iter_start}, max_iters={max_iters}, total_steps={total_steps}")

for iteration in range(iter_start, max_iters+1):
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        step = batch_idx + 1
        # Evaluation
        if step % eval_interval == 0 or step == len(train_loader):
            losses = estimate_loss(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
            )

            train_losses.append(losses['train'])
            val_losses.append(losses['val'])

            print(f"{now()} iteration={iteration:03}/{max_iters:03}, step={step:07_}/{total_steps:07_}, ", end='')
            print(f"train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}")

        # Training step
        logits, loss = model(x_batch, y_batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        #batch_loss = loss.item()

    checkpoint_prefix = str(checkpoint_dir / f"checkpoint_{iteration:06}")

    losses = estimate_loss(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
    )

    meta = {
        'created_at': now(),
        'parameters': parameters,
        "epoch": iteration,
        'train_loss': float(losses['train']),
        'validation_loss': float(losses['val']),
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    with open(checkpoint_prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=4)

    torch.save(checkpoint, checkpoint_prefix+".pt")
    print(f"{now()} saved checkpoint: {checkpoint_prefix}.pt")

--> Start training: iter_start=61, max_iters=60, total_steps=493


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()